# The torus parametrization of Fibonacci quasicrystal
An interactive notebook

**Mykhailo Koreshkov**  
_Department of Mathematical Physics, Institute of Mathematics of the NAS of Ukraine_  
mykhailo.koreshkov@gmail.com

31 March 2026


### References
- M. Baake, J. Hermisson and P. A. B. Pleasants, The torus parametrization of quasiperiodic LI-classes, J. Phys. A **30** (1997), no. 9, 3029-3056.
- M. Schlottmann, Generalized model sets and dynamical systems, in *Directions in mathematical quasicrystals*, 143-159, CRM Monogr. Ser., 13, Amer. Math. Soc., Providence, RI.
- R. V. Moody, M. O. Nesterenko, J. Patera, Computing with almost periodic functions, Acta Crystallogr. Sect. A **64** (2008), no. 6, 654-669.



### Imports

In [ ]:
# I am using this opportunity to recommend uv package manager (https://docs.astral.sh/uv/)
# uv venv
# uv pip install ipykernel ipywidgets matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt

### Details

Axes are standard, physical space is sloped.

Windows is a projection of the Voronoi cell around the origin onto the internal space.
For standard lattice Voronoi cell and fundamental cell are equal.

Window is symmetric.

Chain is singular for t = (0.5, 0.5).

Green point on the plot is `t` -- the origin of the model set and the value for torus parametrization of local hull / LI-class of these Fibonacci chains.

## Setup
Run whole block to proceed

In [ ]:
def calc_W_voronoi(ep, ei):
    """ 
    Assume lattice is rectangular with basis vectors ex, ey.
    Project fundamental region centered at the origin onto internal space = span<ei> and find the window W.
    Returns: tuple(W_min, W_max). W = [W_min, W_max], W_min <= W_max
    """
    p1 = (-0.5, 0.5)
    p2 = (0.5, -0.5)
    w0 = np.dot(p1, ei) / np.linalg.norm(ei)
    w1 = np.dot(p2, ei) / np.linalg.norm(ei)
    return (min(w0, w1), max(w0, w1))

In [ ]:
def _line(ax, p1, p2, **kwargs):
    """Helper function to plot a line between two points p1 and p2."""
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], **kwargs)

def plot_E(ax, ep, ei, R: float = 10, **kwargs):
    """Plot the Ephys and Eint spaces from its basis vectors. Mark the basis vectors with arrows."""
    p0 = ep*(-2*R)
    p1 = ep*2*R
    i0 = ei*(-2*R)
    i1 = ei*2*R
    _line(ax, p0, p1, **kwargs)
    _line(ax, i0, i1, **kwargs)
    ax.arrow(0, 0, ep[0], ep[1], head_width=0.2, head_length=0.3, fc='blue', ec='blue', length_includes_head=True) #angles='xy' ?
    ax.arrow(0, 0, ei[0], ei[1], head_width=0.2, head_length=0.3, fc='orange', ec='orange', length_includes_head=True)
    
def plot_W(ax, ep, ei, W, R: float = 10, fill=False):
    """Plot the window W as a horizontal strip around ep line between W[0] and W[1]"""
    normal = ei / np.linalg.norm(ei)
    w0 = normal * W[0]
    w1 = normal * W[1]
    w00 = w0 - ep * 2 * R
    w01 = w0 + ep * 2 * R
    w10 = w1 - ep * 2 * R
    w11 = w1 + ep * 2 * R
    _line(ax, w00, w01, color='blue', linestyle='-', linewidth=0.5)
    _line(ax, w10, w11, color='blue', linestyle='-', linewidth=0.5)
    if fill:
        # makes sense to add a patch instead of filling between two lines, because the lines can be slanted.
        from matplotlib.patches import Polygon
        polygon = Polygon([w00, w01, w11, w10], color='blue', alpha=0.05)
        ax.add_patch(polygon)   

def make_lattice(N):
    """Return a tuple of X, Y coordinates."""
    n = np.arange(-N, N+1)
    a = np.array([1,0])
    b = np.array([0,1])
    points = n[:, np.newaxis, np.newaxis] * a + n[np.newaxis, :, np.newaxis] * b
    return points[..., 0].flatten(), points[..., 1].flatten()

def plot_lattice(ax, L, **kwargs):
    X, Y = L
    ax.scatter(X, Y, **kwargs)
    # ax.set_xlim(-N, N)
    # ax.set_ylim(-N, N)
    ax.set_aspect('equal')
    # ax.grid(True)   

def plot_fundamental_region_xy(ax, t = (0,0), **kwargs):
    """plot square based on cartesian (1,0) (0,1) basis vectors, with bottom left corner at t."""
    d = 1
    p00 = (t[0], t[1])
    p01 = (t[0]+d, t[1])
    p10 = (t[0], t[1]+d)
    p11 = (t[0]+d, t[1]+d)
    from matplotlib.patches import Polygon
    polygon = Polygon([p00, p01, p11, p10], **kwargs)
    ax.add_patch(polygon)

def plot_fundamental_region_uv(ax, ep, ei, t = (0,0), **kwargs):
    """plot square based on ep, ei basis vectors, with bottom left corner at t."""
    d = 1
    p00 = np.array(t, dtype=float)
    p01 = p00 + ep * d
    p10 = p00 + ei * d
    p11 = p00 + ep * d + ei * d
    from matplotlib.patches import Polygon
    polygon = Polygon([p00, p01, p11, p10], **kwargs)
    ax.add_patch(polygon)

def calc_chain(ep, ei, W, t, N=10, eps=1e-6, R=None):
    """
    Compute a chain of lattice points contained in the strip W. Assume standard rectangular lattice starting at `t`.

    The function starts at lattice point `t` and iteratively advances in:
    - positive direction using steps `a=(1,0)` / `b=(0,1)`
    - negative direction using steps `-a` / `-b`

    At each step, candidate points are accepted only if their projection onto the
    internal space lies inside the window interval `W` with tolerance `eps`.

    Function returns two chains of accepted points in the positive and negative directions, and a code sequence for this chain.
    1. step along `dx` -> code 'a'
    2. step along `dy` -> code 'b'
    3. if neither valid, diagonal `dx+dy` -> code 'c'
    If both 1 and 2 are valid, both may be appended and code '?' is emitted (singular chain).
    The sequence code starts with reversed negative chain, then '|' for the starting point, then positive chain.

    Parameters
    ----------
    ep : np.ndarray
        Physical space basis vector.
    ei : np.ndarray
        Internal space basis vector; used for projection onto internal space.
    W : tuple[float, float]
        Window bounds `(W_min, W_max)` in internal-space coordinates.
    t : array-like
        Starting point (not necessarily a lattice point).
    N : int, default=10
        Number of iterations for each direction.
    eps : float, default=1e-6
        Tolerance for window inclusion tests. (especially important for singular chains)
    R : float | None, default=None
        Optional coordinate cutoff. Iteration stops early if any coordinate of
        the current point leaves `[-R, R]`.

    Returns
    -------
    pos_chain : list[np.ndarray]
        Accepted points while iterating in the positive direction. Starts with `t` and extends forward.
    neg_chain : list[np.ndarray]
        Accepted points while iterating in the negative direction. Starts with `t` and extends backward.
    code : list[str]
        Symbolic step sequence:
        reversed(negative codes) + ['|'] + positive codes.
    """
    d = 1
    normal = ei / np.linalg.norm(ei)
    p0 = t
    a = np.array([1,0])
    b = np.array([0,1])
    def _is_valid(p):
        # check if p \in \tilde L maps to a window W in E_int
        return np.dot(p, normal) >= W[0] - eps and np.dot(p, normal) <= W[1] + eps
    def _iterate_chain_from(p0, dx, dy):
        points = [p0]
        code = []
        p = p0.copy()
        for n in np.arange(N+1):
            # positive direction
            p1 = p + dx * d
            p2 = p + dy * d
            p3 = p + (dx + dy) * d
            found_1 = False
            found_2 = False
            found_3 = False
            if _is_valid(p1):
                points.append(p1)
                p = p1
                found_1 = True
            if _is_valid(p2):
                points.append(p2)
                p = p2
                found_2 = True
                if found_1:
                    print(f"Warning: both directions are valid for point {p}, choosing {p2} over {p1}")
            if not found_1 and not found_2 and _is_valid(p3):
                points.append(p3)
                p = p3
                found_3 = True
            if found_1 and found_2:
                code.append('?')
            elif found_1:
                code.append('a')
            elif found_2:
                code.append('b')
            elif found_3:
                code.append('c')
            if R is not None and (p.max() > R or p.min() < -R):
                break 
        return points, code
    pos_chain, pos_code = _iterate_chain_from(p0, a, b)
    neg_chain, neg_code = _iterate_chain_from(p0, -a, -b)
    return pos_chain, neg_chain, list(reversed(neg_code)) + ['|'] + pos_code

def plot_chain(ax, chain, **kwargs):
    X, Y = np.array(chain).T
    ax.plot(X, Y, **kwargs)

In [ ]:
def map_uv_to_xy(ei, ep, u, v):
    """
    uv coordinates are in the basis of (ep, ei). This is useful because translation by (1,0) gives the same chain.
    xy coordinates are in the standard basis of (1,0) and (0,1). 
    To remain in the fundamental region, we can take u and v modulo 1.
    """
    xy = u * ep + v * ei
    xy = xy % 1
    return xy
    
def map_xy_to_uv(ei, ep, x, y):
    # inverse of map_uv_to_xy
    A = np.array([ep, ei]).T
    uv = np.linalg.solve(A, [x, y])
    return uv

In [ ]:
def project_point_onto_line(p, line_dir, line_point=(0,0)):
    line_dir = line_dir / np.linalg.norm(line_dir)
    return line_point + np.dot(p - line_point, line_dir) * line_dir

In [ ]:
face_colors = ['red', 'blue']
def plot_translated_chain(R, N, uv=None, xy=None, 
                          show_chain=True, 
                          show_projected_points=False, 
                          use_xy_fundamental_region=True, 
                          show_projection_lines=False,
                          show_projected_faces=False):
    """
    Visualize a translated Fibonacci cut-and-project set (model set) in 2D by connecting feasible points to form a "chain".

    Builds a square lattice, applies a translation, selects points inside the strip
    (window) defined by the internal-space projection, and optionally draws:
    - the "chain" in physical/internal embedding space,
    - orthogonal projections of chain points onto the physical-space line,
    - projection helper lines,
    - colored projected edge segments ("faces").

    You can use either `uv` or `xy` translation coordinates, but not both simultaneously. 
    The `uv` coordinates are in the basis of (ep, ei) -- the physical and internal space basis vectors. 
    The `xy` coordinates are in the standard Cartesian basis.

    Parameters
    ----------
    R : float
        Plot radius and clipping bound for axes/chains.
    N : int
        Lattice half-size for `make_lattice`; points are generated on [-N, N]^2.
    uv : tuple[float, float] | None, default None
        Translation in (u, v) coordinates (basis: ep, ei).
    xy : tuple[float, float] | None, default None
        Translation in Cartesian (x, y) coordinates.
    show_chain : bool, default True
        If True, draw positive and negative chain polylines.
    show_projected_points : bool, default False
        If True, project chain points onto E_phys and draw projected points.
    use_xy_fundamental_region : bool, default True
        If True, draw unit square in xy basis; otherwise draw fundamental region in uv basis.
    show_projection_lines : bool, default False
        If True and projected points are shown, draw lines from chain points to projections.
    show_projected_faces : bool, default False
        If True and projected points are shown, color projected segments by edge direction
        using global `face_colors`.

    Returns
    -------
    pos_chain : list[np.ndarray]
        Chain of points in the positive direction from the starting point.
    neg_chain : list[np.ndarray]
        Chain of points in the negative direction from the starting point.
    code : list[str]
        Symbolic representation of the chain.
    
    Produces matplotlib graphics and prints diagnostic information.
    """
    tau = (1 + np.sqrt(5)) / 2
    tau1 = -1/tau
    invtau = 1/tau
    print(f"tau = {tau:.3f}, tau' = {tau1:.3f}, 1/tau = {invtau:.3f}")

    # Physical space Ep = span<ep>. Internal space Ei = span<ei>. <ep, ei> = 0.
    ep = np.array([1, invtau])
    ei = np.array([tau1, 1])

    # translation
    assert not (uv is not None and xy is not None), "Please, do not use uv and xy simultaneously"
    if uv is not None:
        t = map_uv_to_xy(ei, ep, *uv)
    if xy is not None:
        t = np.array(xy)
        uv = map_xy_to_uv(ei, ep, *xy)
    print(f"uv coordinates: {uv[0]:.6f}, {uv[1]:.6f}")
    print(f"xy coordinates: {t[0]:.6f}, {t[1]:.6f}")

    W = calc_W_voronoi(ep, ei)
    W = (W[0] - 5e-6, W[1] + 5e-6) # add some padding to the window
    # W = (W[0] + 0.1, W[1] - 0.1) # remove some padding to the window
    print(f"W = {W[0]:.3f}, {W[1]:.3f}")

    fig, ax = plt.subplots(figsize=(6, 6))
    plot_E(ax, ep, ei, R=R, color='black', linewidth=0.5)
    plot_W(ax, ep, ei, W, R=R, fill=True)

    L = make_lattice(N=N)
    L = L[0] + t[0], L[1] + t[1]

    plot_lattice(ax, L, s=3, c='black')
    if use_xy_fundamental_region:
        plot_fundamental_region_xy(ax, t=(0,0), color='black', alpha=0.1)
    else:
        plot_fundamental_region_uv(ax, ep, ei, t=(0,0), color='black', alpha=0.1)

    pos_chain, neg_chain, chain_code = calc_chain(ep, ei, W, t=t, N=N, R=R)
    print(f"pos_chain: {len(pos_chain)} points, neg_chain: {len(neg_chain)} points")
    if show_chain:
        plot_chain(ax, pos_chain, color='black', linewidth=1)
        plot_chain(ax, neg_chain, color='black', linewidth=1)
    print("CHAIN")
    print("".join(chain_code))

    if show_projected_points:
        xy_chain = list(reversed(neg_chain)) + pos_chain[1:]
        projected_chain = [project_point_onto_line(p, ep, (0,0)) for p in xy_chain]
        if show_projection_lines:
            for p, pp in zip(xy_chain, projected_chain):
                ax.plot([p[0], pp[0]], [p[1], pp[1]], color='gray', linestyle='-', linewidth=0.5)
        X, Y = np.array(projected_chain).T
        ax.scatter(X, Y, s=10, color='red')
        if show_projected_faces:
            # face_color_id = 0
            for p0,p1,pp0,pp1 in zip(xy_chain, xy_chain[1:], projected_chain, projected_chain[1:]):
                face_color_id = int(p1[0] > p0[0]) # color faces based on the direction of the edge
                plt.plot([pp0[0], pp1[0]], [pp0[1], pp1[1]], color=face_colors[face_color_id], linestyle='-', linewidth=2)
                # face_color_id = (face_color_id + 1) % len(face_colors)

    ax.scatter([t[0]], [t[1]], s=50, color='green', label='translation point')

    ax.set_xlim(-R, R)
    ax.set_ylim(-R, R)

    return pos_chain, neg_chain, chain_code

## Some examples

In [ ]:
TAU = (1 + np.sqrt(5)) / 2
TAU1 = -1/TAU
INVTAU = 1/TAU
print(f"tau = {TAU:.3f}, tau' = {TAU1:.3f}, 1/tau = {INVTAU:.3f}")

# Physical space Ep = span<ep>. Internal space Ei = span<ei>. <ep, ei> = 0.
ep = np.array([1, INVTAU])
ei = np.array([TAU1, 1])

In [ ]:
u,v = (1.3, 0)
_, _, code1 = plot_translated_chain(R=8, N=10, uv=(u, v), show_chain=True, 
                      show_projected_points=True, 
                      show_projection_lines=True,
                      show_projected_faces=False)
plt.savefig(f"fibonacci_a_{u}_{v}.png", dpi=150, bbox_inches='tight')

In [ ]:
_, _, code1 = plot_translated_chain(R=8, N=10, uv=(0, 0), show_chain=True, 
                      show_projected_points=True, 
                      show_projection_lines=True,
                      show_projected_faces=True)


Translate by 0.5 in physical space.

In [ ]:
_, _, code2 = plot_translated_chain(R=8, N=10, uv=(0.5, 0), 
                      show_projected_points=True, 
                      show_projection_lines=True,
                      show_projected_faces=True)
# plt.gcf().savefig("classic_fibonacci_1.png", dpi=150)
# plt.grid(True)

In [ ]:
code1 == code2

Torus behavious: **v** direction wraps around and starts in the bottom right corner of the fundamental region 

Different **v** values produce "different" sequences around zero.

However, all sequences consist of the same two (three for specific values) tiles: a long and a short one.

In [ ]:
plot_translated_chain(R=4.5, N=10, uv=(0, 0.1),
                      show_projected_points=True, 
                      show_projection_lines=True,
                      show_projected_faces=True);

Singular chain: three tile sizes and ambiguous code

In [ ]:
plot_translated_chain(R=4.5, N=10, xy=(0.5, 0.5), show_chain=True, show_projected_points=True);
plt.savefig("fibonacci_xy_0.5_0.5.png", dpi=150, bbox_inches='tight')

## Visualization


- Drag the slider to select uv or xy coordinates or click on the value to the right and enter a new one
- Watch the model set change or translate

### UV coords

In [ ]:
import ipywidgets as widgets
from IPython.display import display

u_slider = widgets.FloatSlider(
    value=0.0, min=-2.0, max=2.0, step=0.001,
    description="u", readout_format=".3f", continuous_update=False
)
v_slider = widgets.FloatSlider(
    value=0.0, min=-2.0, max=2.0, step=0.001,
    description="v", readout_format=".3f", continuous_update=False
)
controls = widgets.VBox([u_slider, v_slider])

def _update(u, v):
    plot_translated_chain(R=5, N=20, uv=(u, v), show_chain=False, show_projected_points=True)
    plt.show()

out = widgets.interactive_output(_update, {"u": u_slider, "v": v_slider})
display(controls, out)

### XY coords

In [ ]:
import ipywidgets as widgets
from IPython.display import display

x_slider = widgets.FloatSlider(
    value=0.0, min=0.0, max=1.0, step=0.001,
    description="x", readout_format=".3f", continuous_update=False
)
y_slider = widgets.FloatSlider(
    value=0.0, min=0.0, max=1.0, step=0.001,
    description="y", readout_format=".3f", continuous_update=False
)
controls = widgets.VBox([x_slider, y_slider])

def _update(x, y):
    plot_translated_chain(R=10, N=20, xy=(x, y), show_chain=True, show_projected_points=True)
    plt.show()

out = widgets.interactive_output(_update, {"x": x_slider, "y": y_slider})
display(controls, out)

### Only $\Lambda(x,y)$ Points

In [ ]:
import ipywidgets as widgets
from IPython.display import display

x_slider = widgets.FloatSlider(
    value=0.0, min=0.0, max=1.0, step=0.001,
    description="x", readout_format=".3f", continuous_update=False
)
y_slider = widgets.FloatSlider(
    value=0.0, min=0.0, max=1.0, step=0.001,
    description="y", readout_format=".3f", continuous_update=False
)
controls = widgets.VBox([x_slider, y_slider])

def _update(x, y):
    plot_translated_chain(R=5, N=20, xy=(x, y), show_chain=False, show_projected_points=True)
    plt.show()

out = widgets.interactive_output(_update, {"x": x_slider, "y": y_slider})
display(controls, out)